In [56]:
import requests
import time
import json
import os
from datetime import datetime

In [57]:
from pyspark.sql.functions import lit

In [58]:
from pyspark.sql import SparkSession

In [59]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

In [60]:
ENDPOINTS = [
    "/api/v1/data/orders",
    "/api/v1/data/customers",
    "/api/v1/data/order_items",
    "/api/v1/data/products",
    "/api/v1/data/sellers",
    "/api/v1/data/payments"
]

In [61]:
import os

API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

PAGE_SIZE = 1000
MAX_RETRIES = 5

os.makedirs("bronze", exist_ok=True)
os.makedirs("quarantine", exist_ok=True)

In [62]:
class APIClient:
    def __init__(self):
        self.token = self.authenticate()

    def authenticate(self):
        resp = requests.post(
            f"{API_BASE_URL}/api/v1/auth/token",
            json={"username": API_USERNAME, "password": API_PASSWORD}
        )
        return resp.json()["access_token"]

    def get_headers(self):
        return {"Authorization": f"Bearer {self.token}"}

    def refresh_token(self):
        print("Refreshing token...")
        self.token = self.authenticate()

In [63]:
def make_request(client, url, params):
    retries = 0

    while retries < MAX_RETRIES:
        response = requests.get(url, headers=client.get_headers(), params=params)

        if response.status_code == 200:
            return response

        elif response.status_code == 401:
            client.refresh_token()

        elif response.status_code in [429, 500]:
            wait = 2 ** retries
            print(f"Retry {retries+1}, waiting {wait}s...")
            time.sleep(wait)
            retries += 1

        else:
            print(f"Failed: {response.status_code}")
            print(response.text)
            return None

    return None

In [64]:
MANIFEST_FILE = "manifest.json"

def load_manifest():
    if not os.path.exists(MANIFEST_FILE):
        return {}
    with open(MANIFEST_FILE, "r") as f:
        return json.load(f)

def save_manifest(manifest):
    with open(MANIFEST_FILE, "w") as f:
        json.dump(manifest, f, indent=2)

In [65]:
def ingest_endpoint_spark(client, endpoint):
    manifest = load_manifest()
    ingested_pages = set(manifest.get(endpoint, []))

    page = 1

    while True:
        if page in ingested_pages:
            print(f"Skipping page {page}")
            page += 1
            continue

        response = make_request(
            client,
            f"{API_BASE_URL}{endpoint}",
            {"page": page, "page_size": PAGE_SIZE}
        )

        if not response:
            break

        json_data = response.json()

        # Validate
        if "data" not in json_data:
            print(f"Malformed response page {page}")
            with open(f"quarantine/{endpoint.replace('/', '_')}_{page}.json", "w") as f:
                json.dump(json_data, f)
            break

        records = json_data["data"]

        if not records:
            print(f"Empty page {page}")
            break

        df = spark.createDataFrame(records)


        df = df.withColumn("_ingested_at", lit(datetime.utcnow().isoformat()))
        df = df.withColumn("_source_endpoint", lit(endpoint))

        output_path = f"bronze/{endpoint.split('/')[-1]}"

        df.write \
          .mode("append") \
          .option("header", "true") \
          .csv(output_path)

        print(f"Page {page} written ({len(records)} records)")

        ingested_pages.add(page)

        if not json_data["pagination"]["has_next"]:
            break

        page += 1

    # Save manifest
    manifest[endpoint] = list(ingested_pages)
    save_manifest(manifest)

In [66]:
client = APIClient()

for endpoint in ENDPOINTS:
    print(f"Processing {endpoint}")
    ingest_endpoint_spark(client, endpoint)

Processing /api/v1/data/orders
Skipping page 1
Skipping page 2
Skipping page 3
Skipping page 4
Skipping page 5
Skipping page 6
Skipping page 7
Skipping page 8
Skipping page 9
Skipping page 10
Skipping page 11
Skipping page 12
Skipping page 13
Skipping page 14
Skipping page 15
Skipping page 16
Skipping page 17
Skipping page 18
Skipping page 19
Skipping page 20
Skipping page 21
Skipping page 22
Skipping page 23
Skipping page 24
Skipping page 25
Skipping page 26
Skipping page 27
Skipping page 28
Skipping page 29
Skipping page 30
Skipping page 31
Skipping page 32
Skipping page 33
Skipping page 34
Skipping page 35
Skipping page 36
Skipping page 37
Skipping page 38
Skipping page 39
Skipping page 40
Skipping page 41
Skipping page 42
Skipping page 43
Skipping page 44
Skipping page 45
Skipping page 46
Skipping page 47
Skipping page 48
Skipping page 49
Skipping page 50
Skipping page 51
Skipping page 52
Skipping page 53
Skipping page 54
Skipping page 55
Skipping page 56
Skipping page 57
Skipping 